# Real Data ST Column Lag-Cap Parameter Check

Fit only Column Vecchia models on one real day, changing only the temporal lag conditioning caps:

- `(14, 14, 14)`: full same-time, t-1, t-2 reverse-L conditioning
- `(14,  6,  6)`: full same-time, but only nearest 6 candidates from t-1 and t-2
- `(14,  0,  0)`: same-time reverse-L only, no temporal conditioning

Goal: check whether removing temporal conditioning makes `range_lat`, `range_lon`, `range_time`, and `nugget` inflate. That directly tests the idea that temporal/advection dependence, if not conditioned on, is reinterpreted as longer spatial range.


In [1]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_st_lagcap_050726 import ColumnSTLagCapTrendVecchiaFit

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
torch: 2.5.1
device: cpu


In [2]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]
MEAN_DESIGN = 'hour_spatial'

COLUMN_VARIANTS = [
    {
        'variant': 'column_lags_14_14_14',
        'model': 'ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc',
        'per_lag_conditioning_counts': (14, 14, 14),
        'note': 'full same-time/t-1/t-2 reverse-L conditioning',
    },
    {
        'variant': 'column_lags_14_06_06',
        'model': 'ColumnSTLagCap_14_06_06_Up2_Right3_head0_realloc',
        'per_lag_conditioning_counts': (14, 6, 6),
        'note': 'full same-time; only nearest 6 from each temporal lag',
    },
    {
        'variant': 'column_lags_14_00_00',
        'model': 'ColumnSTLagCap_14_00_00_Up2_Right3_head0_realloc',
        'per_lag_conditioning_counts': (14, 0, 0),
        'note': 'same-time reverse-L only; no temporal conditioning',
    },
]

COLUMN_COMMON = {
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'include_lag_self': False,
    'target_chunk_size': 512 if DEVICE.type == 'cpu' else 4096,
}

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_RESULTS_CSV = OUT_DIR / 'real_st_column_lagcap_params_050726_results.csv'
OUT_PARAMS_CSV = OUT_DIR / 'real_st_column_lagcap_params_050726_param_table.csv'

print('variants:', [v['variant'] for v in COLUMN_VARIANTS])
print('output:', OUT_RESULTS_CSV)


variants: ['column_lags_14_14_14', 'column_lags_14_06_06', 'column_lags_14_00_00']
output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/real_st_column_lagcap_params_050726_results.csv


In [3]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def st_diag(model):
    batched = getattr(model, 'Batched_Groups', None)
    if not batched:
        return {'n_batches': 0, 'n_tails': 0, 'mean_m': np.nan, 'max_m': np.nan, 'largest_batch_n': np.nan}
    group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in batched], dtype=np.int64)
    m_sizes = np.asarray([int(g['max_m']) for g in batched], dtype=np.int64)
    return {
        'n_batches': int(len(batched)),
        'n_tails': int(group_sizes.sum()),
        'mean_m': float(m_sizes.mean()),
        'max_m': int(m_sizes.max()),
        'largest_batch_n': int(group_sizes.max()),
    }


def save_outputs(rows):
    results = pd.DataFrame(rows)
    round_df(results).to_csv(OUT_RESULTS_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    param_rows = []
    for _, row in results.iterrows():
        for p in P_LABELS:
            param_rows.append({
                'date_str': row['date_str'],
                'mean_design': row['mean_design'],
                'variant': row['variant'],
                'model': row['model'],
                'parameter': p,
                'estimate': row[f'est_{p}'],
            })
    param_table = pd.DataFrame(param_rows)
    round_df(param_table).to_csv(OUT_PARAMS_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    return results, param_table

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])


Initial log params: [3.9558, 1.3863, 0.4463, -3.5835, 0.0218, -0.1689, -1.3984]


In [4]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)

del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')


--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Month time slots loaded then trimmed: 248 -> 8
Grid cells: 18,126


In [5]:
def load_day_map(day_idx, keep_ori=True):
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=keep_ori,
    )
    return {k: v.to(DEVICE) for k, v in day_map.items()}, day_keys


def fit_column_variant(day_idx, smooth, spec):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\\n' + '=' * 90)
    print(f"COLUMN FIT | {spec['variant']} | DAY {date_str} | mean={MEAN_DESIGN} | valid={n_valid:,}")
    print('caps:', spec['per_lag_conditioning_counts'], '|', spec['note'])

    params = make_params()
    model = ColumnSTLagCapTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=COLUMN_COMMON['head_right_cols'],
        above_count=COLUMN_COMMON['above_count'],
        right_col_count=COLUMN_COMMON['right_col_count'],
        per_lag_conditioning_counts=spec['per_lag_conditioning_counts'],
        include_lag_self=COLUMN_COMMON['include_lag_self'],
        target_chunk_size=COLUMN_COMMON['target_chunk_size'],
        use_data_coords_for_offsets=True,
        mean_design=MEAN_DESIGN,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_params(out)

    row = {
        'date_str': date_str,
        'smooth': smooth,
        'mean_design': MEAN_DESIGN,
        'variant': spec['variant'],
        'model': spec['model'],
        'lag_cap_t': spec['per_lag_conditioning_counts'][0],
        'lag_cap_tminus1': spec['per_lag_conditioning_counts'][1],
        'lag_cap_tminus2': spec['per_lag_conditioning_counts'][2],
        'loss': float(out[-1]),
        'fit_iter_raw': int(fit_iter),
        'precompute_s': pre_s,
        'fit_s': fit_s,
        'total_s': pre_s + fit_s,
        'n_valid': n_valid,
        'note': spec['note'],
    }
    row.update(diag)
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    print('RESULT:', {k: row[k] for k in ['variant', 'loss', 'est_range_lat', 'est_range_lon', 'est_range_time', 'est_nugget', 'est_advec_lon']})
    del model, day_map, params, opt
    empty_device_cache()
    return row


In [6]:
rows = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        for spec in COLUMN_VARIANTS:
            rows.append(fit_column_variant(day_idx, smooth, spec))
            results, param_table = save_outputs(rows)

print('Saved results:', OUT_RESULTS_CSV)
print('Saved params:', OUT_PARAMS_CSV)

display_cols = [
    'date_str', 'variant', 'lag_cap_t', 'lag_cap_tminus1', 'lag_cap_tminus2',
    'loss', 'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'mean_m', 'max_m', 'n_tails', 'precompute_s', 'fit_s', 'total_s',
]
display(round_df(results[display_cols]))
display(round_df(param_table))


\n==========================================================================================
COLUMN FIT | column_lags_14_14_14 | DAY 20240703 | mean=hour_spatial | valid=140,352
caps: (14, 14, 14) | full same-time/t-1/t-2 reverse-L conditioning
Pre-computing Batched ReverseLColumn LagCap V3 [heads_right=0, above=2, right_cols=3, lag_caps=(14, 14, 14), coord_mode=data]... Done in 2.1s. grid=114x159, heads=0, tails=140352, batches=[(14, 18067), (28, 17670), (42, 104615)], m mean/med/max=36.0/42/42
--- Starting Batched Reverse-L Vecchia L-BFGS ---
--- Step 1/5 / Loss: 1.470097 / Max Grad: 2.19e-03 ---
--- Step 2/5 / Loss: 1.383566 / Max Grad: 2.21e-04 ---
--- Step 3/5 / Loss: 1.382125 / Max Grad: 1.51e-04 ---
--- Step 4/5 / Loss: 1.381834 / Max Grad: 8.93e-06 ---
Converged: max_grad 8.93e-06 < 1.00e-05
Final Interpretable Params: {'sigma_sq': 12.416294211885816, 'range_lon': 0.1698206737652684, 'range_lat': 0.14368913583652082, 'range_time': 1.2903612852843496, 'advec_lat': -0.06828545853

,date_str,variant,lag_cap_t,lag_cap_tminus1,lag_cap_tminus2,loss,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget,mean_m,max_m,n_tails,precompute_s,fit_s,total_s
0,20240703,column_lags_14_14_14,14,14,14,1.3818,12.4163,0.1437,0.1698,1.2904,-0.0683,-0.2343,1.2488,28.0,42,140352,2.1810,273.5094,275.6904
1,20240703,column_lags_14_06_06,14,6,6,1.3819,12.4250,0.1437,0.1701,1.3501,-0.0777,-0.2321,1.2496,20.0,26,140352,1.8193,132.8576,134.6769
2,20240703,column_lags_14_00_00,14,0,0,1.3911,13.7581,0.1692,0.2013,1.2077,0.0218,-0.1689,1.3938,14.0,14,140352,1.4707,22.2186,23.6894


,date_str,mean_design,variant,model,parameter,estimate
0,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,sigmasq,12.4163
1,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,range_lat,0.1437
2,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,range_lon,0.1698
3,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,range_time,1.2904
4,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,advec_lat,-0.0683
5,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,advec_lon,-0.2343
6,20240703,hour_spatial,column_lags_14_14_14,ColumnSTLagCap_14_14_14_Up2_Right3_head0_realloc,nugget,1.2488
7,20240703,hour_spatial,column_lags_14_06_06,ColumnSTLagCap_14_06_06_Up2_Right3_head0_realloc,sigmasq,12.4250
8,20240703,hour_spatial,column_lags_14_06_06,ColumnSTLagCap_14_06_06_Up2_Right3_head0_realloc,range_lat,0.1437
9,20240703,hour_spatial,column_lags_14_06_06,ColumnSTLagCap_14_06_06_Up2_Right3_head0_realloc,range_lon,0.1701
